# Temporal Leakage & Point-in-Time Joins

Wiki reference for [Temporal Leakage & Point-in-Time Joins](https://ml-viz-ruby.vercel.app/wiki/temporal-leakage). To keep your own copy, use **File -> Save a copy in Drive**.

**The problem.** A feature computed from event logs can embed information that will not exist at serving time. The offline metric looks great; production collapses.

**The idea in one sentence.** For each row a feature must use only data available at that row's prediction time; a naive "read the current table" join hands the model its own future, and only a *point-in-time (as-of)* join reproduces what serving will actually see.

We build a fraud dataset where a transaction is *reversed* 7 days after fraud is confirmed, compute a `was_reversed` feature two ways (naive current-state vs as-of), and watch a near-perfect offline AUC collapse to the honest one on deploy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

plt.style.use('dark_background')
plt.rcParams.update({
    'axes.facecolor': '#1a1d27', 'figure.facecolor': '#0f1117',
    'axes.edgecolor': '#3a3d4a', 'grid.color': '#2a2d3a',
    'text.color': '#e2e8f0', 'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
})
rng = np.random.default_rng(0)

# One transaction per day, t = 0..N-1. Higher-amount transactions are a bit
# more likely to be fraud -- a weak but HONEST signal available at prediction time.
N = 4000
amount = rng.gamma(2.0, 1.0, size=N)
p_fraud = 1 / (1 + np.exp(-(amount - 3.0)))      # sigmoid in amount
fraud = (rng.random(N) < 0.06 * p_fraud / p_fraud.mean()).astype(int)

# A reversal is logged 7 days AFTER a transaction, for 90% of frauds and a rare
# 2% of legitimate transactions (customer disputes).
DELAY = 7
reversed_flag = ((fraud == 1) & (rng.random(N) < 0.90)) | \
                ((fraud == 0) & (rng.random(N) < 0.02))
reversal_time = np.where(reversed_flag, np.arange(N) + DELAY, np.iinfo(np.int64).max)
print('fraud rate:', fraud.mean().round(3), '| reversed:', reversed_flag.mean().round(3))

## 1 - The trap: "read the current table" hands the model the future

The naive feature reads the reversal table as it looks *now* (a batch job run long after the fact): every reversed transaction shows `was_reversed = 1`. But a reversal is logged 7 days **after** the transaction, so at the moment the fraud decision must be made, no transaction is reversed yet. The as-of feature respects that.

In [ ]:
def was_reversed_asof(cutoff):
    # value visible at time `cutoff`: reversal must have been LOGGED by then
    return (reversal_time <= cutoff).astype(float)

t = np.arange(N)
reversed_naive = reversed_flag.astype(float)          # current DB state -> leaks the future
reversed_asof  = was_reversed_asof(t)                 # point-in-time -> always 0 at decision time

print('naive was_reversed (mean):', reversed_naive.mean().round(3))
print('as-of  was_reversed (mean):', reversed_asof.mean().round(3), '  <- nothing is reversed yet')

## 2 - Same model, two feature versions

Both models also get the honest `amount` feature, so the as-of model is not helpless — it just lacks the leaked flag.

In [ ]:
def auc_with(feature):
    X = np.column_stack([amount, feature])
    Xtr, Xte, ytr, yte = train_test_split(X, fraud, test_size=0.3, random_state=0)
    clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    return roc_auc_score(yte, clf.predict_proba(Xte)[:, 1])

auc_leaky = auc_with(reversed_naive)
auc_asof  = auc_with(reversed_asof)
print(f'Offline AUC with NAIVE  was_reversed (leaks future): {auc_leaky:.3f}')
print(f'Offline AUC with AS-OF  was_reversed (honest)      : {auc_asof:.3f}')

The naive feature is nearly the label itself, so offline AUC is almost perfect. The as-of version is constant at decision time and adds nothing beyond `amount` — the honest ceiling for this data.

## 3 - Serving computes the feature as-of, so the offline number was a fantasy

In [ ]:
print(f'What the dashboard promised (naive, offline): {auc_leaky:.3f}')
print(f'What production actually delivers (as-of)     : {auc_asof:.3f}')
print(f'Illusory lift that vanishes on deploy         : {auc_leaky - auc_asof:.3f}')

fig, ax = plt.subplots(figsize=(6.5, 4))
bars = ax.bar(['offline\n(naive join)', 'production\n(as-of / honest)'],
              [auc_leaky, auc_asof], color=['#f43f5e', '#14b8a6'], width=0.6)
ax.axhline(0.5, ls='--', lw=1, color='#475569')
ax.text(1.42, 0.51, 'chance', color='#475569', fontsize=9, ha='right')
ax.set_ylabel('ROC AUC'); ax.set_ylim(0, 1.0)
ax.set_title('Temporal leakage: the offline score is measuring the future')
for b, v in zip(bars, [auc_leaky, auc_asof]):
    ax.text(b.get_x() + b.get_width()/2, v + 0.02, f'{v:.2f}', ha='center', color='#e2e8f0')
plt.tight_layout(); plt.show()

**What to notice.** The model, the data, and the split are identical across the two bars. The only change is *when* the `was_reversed` value is read — the current table vs point-in-time — and that alone manufactures (then destroys) a huge AUC gap.

## 4 - Taxonomy & fixes

| Kind | Mechanism | Catch it with |
|---|---|---|
| Target leakage | Feature is a consequence of the label | "Would I have this at prediction time?" audit |
| Train-test contamination | Test stats leak into preprocessing | Fit transformers inside the CV fold |
| **Temporal leakage** | Feature window crosses prediction time | Time-based split + point-in-time joins |

- **Never** validate a time-series model with a random split; use a forward (time-based) split.
- Build features with **as-of joins**: for each row, read only records timestamped at/before its prediction time.
- **Lag** each feature by the real pipeline latency -- a value that lands after the decision moment is unavailable, even though it is technically "past".

## 5 - Your turn

Implement the as-of lookup and prove it never reads a reversal from the future. Fill in the `# TODO(you)` line.

In [ ]:
def my_asof(cutoff):
    # Return 1.0 where a reversal was LOGGED at a time <= cutoff, else 0.0.
    # TODO(you): compare reversal_time to cutoff (reversals are logged at t+DELAY)
    return np.zeros(N)  # replace this

# must match the reference as-of feature at several decision times
for c in [t, t + 3, t + DELAY - 1]:      # all strictly before the reversal lands
    assert np.array_equal(my_asof(c), was_reversed_asof(c)), 'mismatch'
    assert my_asof(c).sum() == 0, 'a reversal was counted before it was logged!'
# once we wait past the delay, the reversals become visible
assert my_asof(t + DELAY).sum() == reversed_flag.sum()
print('Correct - as-of matches and never peeks past the cutoff.')

<details>
<summary>Solution</summary>

```python
def my_asof(cutoff):
    return (reversal_time <= cutoff).astype(float)
```

`reversal_time` is `t + DELAY` for reversed transactions. Comparing it to `cutoff` returns 1 only once the reversal has actually been logged -- never before. That single `<=` is point-in-time correctness.
</details>

## Key takeaways

- **Temporal leakage = using a value the model would not have at prediction time.** The label may be in the future; a feature may not.
- **Random splits hide it** (both folds share the leak); a **time-based split** withholds the future the way production does.
- **Point-in-time (as-of) joins** keep the training feature equal to the serving feature: read only records at/before the prediction time, and **lag** by pipeline latency.
- A leaky feature's offline metric is not conservative or noisy -- it is *measuring the future*, and it evaporates on deploy.

**Next:** [Feature Engineering](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/01-feature-engineering) and [Deployment Pitfalls](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/02-deployment-pitfalls) for train-serve skew, plus [Walk-Forward Validation](https://ml-viz-ruby.vercel.app/wiki/walk-forward-validation).